# 🧠 Phase-3 Growth Redesign — WikiText-2 α×η_sep Sweep (D=4096)

**Does B′ (row-centered SPPMI "pull-similar") + the energy-native `H_anti` "keep-apart" *write* paradigmatic structure on real text — locally, without the smush?**

Runs `experiments/61` at **D=4096** (the floor's dimension) on WikiText-2, GPU.

- **Mechanism (precommit §GR):** `G_pull = normalize(α·(S'@G) + (1−α)·G)` then `G = normalize(G + η_sep·force/mean|force|)`, `force = H_anti = −α_anti·∇log(d_eff)` — removes the near-rank-1 common-mode that power-iteration onto `S'` injects (the diagnosed smush). Local lateral-inhibition analog of the global SVD; retires the `_apply_repulsion` homunculus.
- **One effective knob:** `α_anti·η_sep` is the only thing that enters → fix `α_anti=1`, sweep **force-normalized `η_sep`** (scale-invariant relative step).
- **Gate (gauge-free, §GR):** para-vs-random specificity CI>0 **AND** `corr(log cooc, drift) CI-hi<0.15` **AND** collapse floor. *The stream-shuffle gauge is retired — it leaks 0.79 for a 2nd-order operator.*
- **Global reference (CTRL-global):** the **SVD-of-SPPMI oracle** (`experiments/62`) = what the *forbidden global* whitening achieves (+0.109 specificity at D=1024). The local H_anti's job is to approach it.
- **Pre-registered null:** no `(α, η_sep)` cell clears the gate → predictive/successor growth ("R3"), NOT more knobs.

Background: [Report 122](https://github.com/Dypatterson/Neuro-AI/blob/consolidation/role-structure/reports/122_phase3_second_order_growth_oracles/report.md).

## 0 · Set the runtime to GPU
**Runtime → Change runtime type → A100/T4 GPU** before running (A100 preferred at D=4096). The sweep is ~20–30 min on a T4.

In [ ]:
!nvidia-smi -L || echo "⚠️  No GPU — set Runtime → GPU." 

In [ ]:
# Private repo? paste a GitHub token (Settings → Developer settings → Tokens, repo scope). "" if public.
GITHUB_TOKEN = ""  #@param {type:"string"}
BRANCH = "consolidation/role-structure"
_host = "github.com/Dypatterson/Neuro-AI.git"
clone_url = f"https://{GITHUB_TOKEN}@{_host}" if GITHUB_TOKEN else f"https://{_host}"
%cd /content
!rm -rf Neuro-AI
!git clone --depth 1 --branch {BRANCH} {clone_url}
%cd /content/Neuro-AI
!git log --oneline -1
!pip -q install datasets

In [ ]:
# Pre-warm WikiText-2 (pure data load — does NOT touch CUDA, keeping the GPU clean for the workers).
from datasets import load_dataset
load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print("✅ WikiText-2 cached.")

## 1 · GPU sanity — does B′ + H_anti reverse the smush on the CUDA path?
A ~1-min planted run on the GPU at a *collapsing* schedule. Confirms the CUDA growth + `repulsion_force` autograd path works and that H_anti lifts d_eff back up while keeping king/queen specific.

In [ ]:
!python experiments/61_phase3_second_order_growth.py \
  --corpus-source synthetic_planted --variants B_prime --D 1024 --W 6 --epochs 20 \
  --alpha0 0.3 --alpha-decay 0.9 --seeds 2 --k-grid 4 \
  --alpha-anti 1.0 --eta-sep 0.05 --gate gauge_free --device cuda \
  --out reports/_sanity_hanti.json

In [ ]:
import json
c = json.load(open("reports/_sanity_hanti.json"))["grid_results"][0]; p = c["planted_probe"]; cf = c["collapse_floor"]
print(f"d_eff_ratio={cf['d_eff_ratio_real']:.3f}  | king/queen={p['drift_king_queen']:+.3f}  distractor={p['drift_king_distractor']:+.3f}  random={p['drift_random_nontarget_mean']:+.3f}")
print("✅ Expect d_eff lifted off the 0.05 collapse floor AND king/queen ≫ random (H_anti composes with the SPPMI pull).")

## 2 · Global reference (CTRL-global) — the SVD-of-SPPMI oracle
What the *forbidden global* whitening achieves on this corpus (the upper bound the local H_anti is trying to approach, locally). Gauge-free, substrate-free.

In [ ]:
import subprocess, json
subprocess.run(["python","experiments/62_exp61_oracles.py","--corpus-source","wikitext",
                "--D","4096","--W","6","--max-vocab","2000","--svd-rank","300",
                "--device","cuda"], check=True, stdout=open("reports/_oracle_d4096.json","w"))
o = json.load(open("reports/_oracle_d4096.json"))
s = o["ORACLE1_svd_of_sppmi"]
print("CTRL-global (SVD-of-SPPMI):  king/queen =", round(s["king_queen_cos"],3),
      "| specificity (para-random) =", round(s["specificity_para_minus_random"]["mean"],4),
      "CI", [round(x,4) for x in s["specificity_para_minus_random"]["ci"]],
      "| SIGNAL_EXISTS =", s["specificity_para_minus_random"]["SIGNAL_EXISTS"])
print("-> the GLOBAL target; the local B'+H_anti sweep tries to reach it without the global step.")

## 3 · The sweep — B′ + H_anti (gauge-free gate), WikiText-2, D=4096
`α_anti=1`, force-normalized **`η_sep ∈ {0, 0.02, 0.05, 0.1, 0.2, 0.4}`** (η_sep=0 = the A0 baseline / the WikiText NULL), growth-pull **`α ∈ {0.05, 0.1, 0.2, 0.3}`** (swept internally), 5 seeds, SimLex≥5 (n≈40). Plus controls: uncentered **B** and first-order **A**.

In [ ]:
import subprocess, json, pathlib, sys
ETA   = [0.0, 0.02, 0.05, 0.1, 0.2, 0.4]
COMMON = ["--corpus-source","wikitext","--pair-source","simlex","--simlex-min-sim","5.0",
          "--D","4096","--W","6","--epochs","20","--seeds","5","--max-vocab","2000",
          "--alpha-grid","0.05,0.1,0.2,0.3","--gate","gauge_free","--device","cuda"]
def run(tag, extra):
    out = f"reports/_sweep_{tag}.json"
    cmd = ["python","experiments/61_phase3_second_order_growth.py", *extra, *COMMON, "--out", out]
    print("RUN", tag, "...", flush=True)
    subprocess.run(cmd, check=True)
    return json.load(open(out))["grid_results"]
cells = []
for eta in ETA:                                   # B′ + H_anti sweep
    cells += run(f"Bprime_eta{eta}", ["--variants","B_prime","--alpha-anti","1.0","--eta-sep",str(eta)])
cells += run("ctrl_B", ["--variants","B"])         # uncentered control (collapses)
cells += run("ctrl_A", ["--variants","A"])         # first-order control
json.dump(cells, open("reports/_sweep_all.json","w"))
print("✅ total cells:", len(cells))

## 4 · Results — the gauge-free gate

In [ ]:
import json
cells = json.load(open("reports/_sweep_all.json"))
o = json.load(open("reports/_oracle_d4096.json")); ref = o["ORACLE1_svd_of_sppmi"]["specificity_para_minus_random"]
print(f"CTRL-global (SVD) specificity reference = {ref['mean']:+.4f}  CI {[round(x,4) for x in ref['ci']]}\n")
hdr = f"{'variant':8} {'α0':>5} {'η_sep':>6} {'gauge-free para−rand (95% CI)':>32} {'>0':>4} {'collapse':>8} {'d_eff_r':>7} {'corr<.15':>8} {'PASS':>5}"
print(hdr); print("-"*len(hdr))
import glob   # η_sep lives in each run's config, so iterate per source file
for f in sorted(glob.glob("reports/_sweep_Bprime_eta*.json")) + ["reports/_sweep_ctrl_B.json","reports/_sweep_ctrl_A.json"]:
    d = json.load(open(f)); eta = d["config"].get("eta_sep"); var = d["config"]["variants"]
    for c in d["grid_results"]:
        h = c["HEADLINE_gauge_free_para_vs_random"]; cf = c["collapse_floor"]; dc = c["decorrelation"]
        hm, ci = h["hierarchical_mean"], h["hierarchical_ci"]
        cis = f"{hm:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]" if hm==hm and ci[0]==ci[0] else "nan"
        print(f"{c['variant']:8} {c['alpha0']:>5} {eta:>6} {cis:>32} {str(h['CI_gt_0']):>4} "
              f"{str(cf['COLLAPSE_OK']):>8} {cf.get('d_eff_ratio_real',float('nan')):>7.3f} "
              f"{str(dc['ci_hi_lt_0p15']):>8} {str(c['VARIANT_PASS']):>5}")
anypass = any(c["VARIANT_PASS"] for c in cells)
print(f"\nANY B′+H_anti CELL PASSES THE GAUGE-FREE GATE: {anypass}")
print("→ PASS = the local growth WRITES paradigmatic structure (approaching the SVD reference) without the smush — Phase-3 foundation cleared.")
print("→ NULL = per §GR pre-registration: local common-mode removal isn't enough → predictive/successor growth (R3), NOT more knobs.")

## 5 · Save results to Drive

In [ ]:
from google.colab import drive, files
import shutil, pathlib, glob
drive.mount("/content/drive")
dst = pathlib.Path("/content/drive/MyDrive/neuro-ai/results/exp61_growth_redesign"); dst.mkdir(parents=True, exist_ok=True)
for f in ["reports/_sweep_all.json","reports/_oracle_d4096.json","reports/_sanity_hanti.json"] + glob.glob("reports/_sweep_*.json"):
    p = pathlib.Path(f)
    if p.exists(): shutil.copy(p, dst/p.name)
files.download("reports/_sweep_all.json")
print("✅ saved to", dst, "— bring _sweep_all.json + _oracle_d4096.json back for the Report 123 writeup.")